In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

# 1) Load data
df = pd.read_csv("spotify_dataset.csv")

# 2) Clean numeric columns stored as text
df["Streams"] = df["Streams"].astype(str).str.replace(",", "", regex=False)
df["Artist Followers"] = df["Artist Followers"].astype(str).str.replace(",", "", regex=False)

numeric_cols_to_fix = [
    "Streams", "Artist Followers", "Popularity", "Danceability", "Energy", "Loudness",
    "Speechiness", "Acousticness", "Liveness", "Tempo", "Duration (ms)", "Valence"
]
for c in numeric_cols_to_fix:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 3) Features + target
feature_cols = [
    "Streams", "Artist Followers", "Genre", "Danceability", "Energy", "Loudness",
    "Speechiness", "Acousticness", "Liveness", "Tempo", "Duration (ms)", "Valence", "Chord"
]
model_df = df[feature_cols + ["Popularity"]].dropna(subset=["Popularity"])

X = model_df[feature_cols]
y = model_df["Popularity"]

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

# 4) Preprocessing + model
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
])

model = Pipeline([
    ("prep", preprocessor),
    ("reg", Ridge(alpha=1.0))
])

# 5) Train/evaluate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Rows used:", len(model_df))
print("MAE:", round(mean_absolute_error(y_test, preds), 3))
print("R2 :", round(r2_score(y_test, preds), 3))

Rows used: 1545
MAE: 10.713
R2 : 0.016
